In [1]:
import serial
import time


PORT = "/dev/tty.usbserial-A5069RR4"   # use tty, not cu
BAUD = 115200

ser = serial.Serial(PORT, BAUD, timeout=1)

time.sleep(2)   # 🔥 CRITICAL: wait for Arduino reset

ser.reset_input_buffer()  # clear garbage

print("Waiting for boot ACK...")
print(ser.readline().decode(errors="ignore").strip())



Waiting for boot ACK...
ENC:0,1


In [ ]:
def read_line(ser, timeout=1.0):
    start = time.time()
    while time.time() - start < timeout:
        if ser.in_waiting:
            line = ser.readline().decode(errors="ignore").strip()
            return line
    return None

def send_cmd(ser, cmd):
    ser.write((cmd + "\n").encode())

    t0 = time.time()
    while time.time() - t0 < 2:
        line = read_line(ser, timeout=1)
        if not line:
            continue
        if line in ["ACK", "ERR"]:
            return line
        # ignore ENC lines

    return None

def test_basic_commands(ser):
    print("\n--- BASIC COMMAND TEST ---")

    assert send_cmd(ser, "F") == "ACK"
    time.sleep(0.5)

    assert send_cmd(ser, "S") == "ACK"
    time.sleep(0.2)

    assert send_cmd(ser, "B") == "ACK"
    time.sleep(0.5)

    assert send_cmd(ser, "S") == "ACK"

    print("✅ Basic commands passed")


def test_speed(ser):
    print("\n--- SPEED TEST ---")

    assert send_cmd(ser, "V:100") == "ACK"
    assert send_cmd(ser, "F") == "ACK"

    time.sleep(1)

    assert send_cmd(ser, "V:200") == "ACK"

    time.sleep(1)

    assert send_cmd(ser, "S") == "ACK"

    print("✅ Speed test passed")


def test_invalid(ser):
    print("\n--- INVALID COMMAND TEST ---")

    resp = send_cmd(ser, "XYZ")
    assert resp == "ERR"

    resp = send_cmd(ser, "V:abc")
    assert resp == "ACK"  # NOTE: toInt() returns 0 → valid

    print("⚠️ Invalid test behavior verified")


def test_watchdog(ser):
    print("\n--- WATCHDOG TEST ---")

    send_cmd(ser, "F")
    print("Waiting for watchdog timeout...")

    time.sleep(1.0)  # >500ms

    print("Robot should STOP automatically now")
    print("✅ Watchdog test done")


def test_ping(ser):
    print("\n--- PING TEST ---")

    for _ in range(5):
        ser.write(b"PING\n")
        print(">>> PING")
        time.sleep(0.2)

    print("✅ Ping test done (no ACK expected)")


def read_enc_stream(ser, duration=3):
    print("\n--- ENCODER STREAM ---")

    start = time.time()
    while time.time() - start < duration:
        line = read_line(ser, timeout=0.2)
        if line and line.startswith("ENC:"):
            print(line)

    print("✅ Encoder stream test done")


def main():
    ser = serial.Serial(PORT, BAUD, timeout=0.1)
    time.sleep(2)  # allow Arduino reset

    print("Waiting for boot ACK...")
    print(read_line(ser, timeout=2))

    test_basic_commands(ser)
    test_speed(ser)
    test_invalid(ser)
    test_ping(ser)
    read_enc_stream(ser)
    test_watchdog(ser)

    ser.close()


# if __name__ == "__main__":
#     main()

In [8]:
# !pip install pyserial
main()

Waiting for boot ACK...
None

--- BASIC COMMAND TEST ---
>>> F
<<<                               0*  4 3*  6)                 <>F


AssertionError: 